# AeroNetra — Model Evaluation

**Purpose:** Evaluate trained models on the VisDrone validation/test set. Compute mAP, precision, recall, and per-class metrics.

**Kaggle Setup:**
1. Add datasets:
   - `aeronetra-visdrone-yolo` (output from notebook 01)
   - `aeronetra-trained-weights` (output from notebook 02) OR upload weights manually
2. Accelerator: **GPU T4 x2**
3. Internet: ON (if ultralytics needs install)

**Outputs:** Evaluation metrics, confusion matrices, PR curves saved to `/kaggle/working/`.

In [ ]:
# ============================================================
# Cell 1: Install & verify environment
# ============================================================
!pip install -q ultralytics

import torch
import ultralytics
from pathlib import Path

print(f"PyTorch:      {torch.__version__}")
print(f"CUDA:         {torch.cuda.is_available()}")
print(f"Ultralytics:  {ultralytics.__version__}")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# ============================================================
# Cell 2: Configuration
# ============================================================

# --- EDIT: path to your YOLO-format dataset ---
DATASET_YAML = Path("/kaggle/input/aeronetra-visdrone-yolo/visdrone_yolo/dataset.yaml")

# --- EDIT: paths to trained weights ---
# Option A: From training notebook output saved as Kaggle dataset
WEIGHTS_DIR = Path("/kaggle/input/aeronetra-trained-weights/best_weights")

# Option B: Specify individual paths
WEIGHT_PATHS = {
    "YOLOv8n": WEIGHTS_DIR / "yolov8n_visdrone_best.pt",
    "YOLO11n": WEIGHTS_DIR / "yolo11n_visdrone_best.pt",
    "RT-DETR-l": WEIGHTS_DIR / "rtdetr_l_visdrone_best.pt",
}

OUTPUT_DIR = Path("/kaggle/working/evaluation")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Verify files exist
print(f"Dataset YAML: {DATASET_YAML} — {'EXISTS' if DATASET_YAML.exists() else 'MISSING'}")
for name, path in WEIGHT_PATHS.items():
    exists = path.exists()
    size = f"({path.stat().st_size / 1024 / 1024:.1f} MB)" if exists else ""
    print(f"  {name:12s}: {'EXISTS' if exists else 'MISSING'} {size}  {path}")

In [ ]:
# ============================================================
# Cell 3: Evaluate all models
# ============================================================
from ultralytics import YOLO, RTDETR
import time
import json

eval_results = {}

for model_name, weights_path in WEIGHT_PATHS.items():
    if not weights_path.exists():
        print(f"\nSkipping {model_name} — weights not found at {weights_path}")
        continue

    print(f"\n{'='*60}")
    print(f"Evaluating: {model_name}")
    print(f"{'='*60}")

    # Load model
    if "rtdetr" in model_name.lower().replace("-", ""):
        model = RTDETR(str(weights_path))
    else:
        model = YOLO(str(weights_path))

    # Run validation
    t0 = time.time()
    metrics = model.val(
        data=str(DATASET_YAML),
        imgsz=640,
        device=DEVICE,
        project=str(OUTPUT_DIR),
        name=model_name.lower().replace("-", "_"),
        plots=True,       # Generate confusion matrix, PR curve, etc.
        save_json=True,   # Save COCO-format results
        verbose=True,
        exist_ok=True,
    )
    eval_time = time.time() - t0

    # Extract key metrics
    result = {
        "model": model_name,
        "mAP50": float(metrics.box.map50),
        "mAP50-95": float(metrics.box.map),
        "precision": float(metrics.box.mp),
        "recall": float(metrics.box.mr),
        "eval_time_s": eval_time,
    }

    # Per-class mAP50 if available
    if hasattr(metrics.box, 'ap50') and metrics.box.ap50 is not None:
        result["per_class_ap50"] = metrics.box.ap50.tolist()

    eval_results[model_name] = result
    print(f"\n  mAP@50:    {result['mAP50']:.4f}")
    print(f"  mAP@50-95: {result['mAP50-95']:.4f}")
    print(f"  Precision: {result['precision']:.4f}")
    print(f"  Recall:    {result['recall']:.4f}")
    print(f"  Time:      {eval_time:.1f}s")

In [ ]:
# ============================================================
# Cell 4: Comparison table
# ============================================================
import pandas as pd

if eval_results:
    df = pd.DataFrame([
        {
            "Model": r["model"],
            "mAP@50": f"{r['mAP50']:.4f}",
            "mAP@50-95": f"{r['mAP50-95']:.4f}",
            "Precision": f"{r['precision']:.4f}",
            "Recall": f"{r['recall']:.4f}",
            "Time (s)": f"{r['eval_time_s']:.1f}",
        }
        for r in eval_results.values()
    ])
    print("\nModel Comparison on VisDrone Validation Set")
    print("="*70)
    print(df.to_string(index=False))
    
    # Save to CSV
    csv_path = OUTPUT_DIR / "model_comparison.csv"
    df.to_csv(csv_path, index=False)
    print(f"\nSaved to: {csv_path}")
else:
    print("No models were evaluated. Check weight paths above.")

In [ ]:
# ============================================================
# Cell 5: Comparison chart
# ============================================================
import matplotlib.pyplot as plt
import numpy as np

if eval_results:
    models = list(eval_results.keys())
    metrics_to_plot = ["mAP50", "mAP50-95", "precision", "recall"]
    
    x = np.arange(len(models))
    width = 0.2
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    for i, metric in enumerate(metrics_to_plot):
        values = [eval_results[m][metric] for m in models]
        bars = ax.bar(x + i * width, values, width, label=metric)
        ax.bar_label(bars, fmt="%.3f", fontsize=8)
    
    ax.set_xlabel("Model")
    ax.set_ylabel("Score")
    ax.set_title("AeroNetra — Model Evaluation on VisDrone")
    ax.set_xticks(x + width * 1.5)
    ax.set_xticklabels(models)
    ax.legend()
    ax.set_ylim(0, 1.05)
    ax.grid(axis="y", alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "model_comparison.png", dpi=150)
    plt.show()

In [ ]:
# ============================================================
# Cell 6: Per-class analysis (if separate mode)
# ============================================================
import yaml

with open(DATASET_YAML) as f:
    ds_cfg = yaml.safe_load(f)

class_names = ds_cfg.get("names", {})
nc = ds_cfg.get("nc", len(class_names))

if nc > 1 and eval_results:
    print(f"Per-class AP@50 ({nc} classes)")
    print("="*70)
    
    rows = []
    for model_name, result in eval_results.items():
        per_class = result.get("per_class_ap50", [])
        if per_class:
            row = {"Model": model_name}
            for i, ap in enumerate(per_class):
                cname = class_names.get(i, str(i))
                row[cname] = f"{ap:.4f}"
            rows.append(row)
    
    if rows:
        df_class = pd.DataFrame(rows)
        print(df_class.to_string(index=False))
        df_class.to_csv(OUTPUT_DIR / "per_class_ap50.csv", index=False)
    else:
        print("Per-class AP not available in results.")
else:
    print("Single-class (merged) mode — per-class breakdown not applicable.")

In [ ]:
# ============================================================
# Cell 7: Save full results as JSON
# ============================================================

# Save complete evaluation results
results_json = OUTPUT_DIR / "evaluation_results.json"
with open(results_json, "w") as f:
    json.dump(eval_results, f, indent=2, default=str)

print(f"Full results saved to: {results_json}")
print(f"\nAll evaluation outputs in: {OUTPUT_DIR}")
print("\nGenerated files:")
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        size_kb = p.stat().st_size / 1024
        print(f"  {p.relative_to(OUTPUT_DIR)}  ({size_kb:.0f} KB)")